# ДЗ 5. Первый взгляд на orders.csv в Pandas
Срок: **до воскресенья 22.11, 24:00**.
Сдача: папка `hw05` репозитория `dsa-homework` (`hw05.ipynb`): `git add hw05` → `git commit` → `git push`, ссылку прикрепите в LMS.

Каждый ответ: **число и вывод для менеджера** (в print или Markdown-ячейке). Менеджер не читает код: вывод отвечает, что видно в данных и что делать.

Инструменты: Pandas с урока 5 и всё, что было раньше. `groupby`, `apply`, `.str`-методы (кроме готовой строки для `amount`) пока не используем. Если берёте непройденный инструмент, объясните его в комментарии.

- **[База]** обязательно для всех · **[Стандарт]** сколько получится · **[Челлендж]** для тех, кому легко.

 ## [Для всех] Чтение кода: без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
import pandas as pd
# 1
s = pd.Series([10, 20, 30], index=["a", "b", "c"])
print(s["b"], s.iloc[0])
# 2
a = pd.Series({"Алматы": 5, "Шымкент": 2})
b = pd.Series({"Шымкент": 1, "Астана": 4})
print(a + b)
# 3
df = pd.DataFrame({"x": [1, 2, 3], "y": [4, 5, 6]})
print(df[df["x"] > 1].shape)
# 4
print(type(df["x"]).__name__, type(df[["x"]]).__name__)
# 5
f = df[df["x"] > 1]
print(f.index.tolist(), f.iloc[0, 1])
```

Ваши ответы и объяснения:

1. iloc uses position therefore, 10, loc uses label therefore 20
2. there is no almaty in b, therefore it became Nan, shymkent became 3 and new row is added as astana also Nan
3. it removes row0, 2x2
4. series, DataFrame
5. we removed 1st row, which was indexed with 0, so indexes that are left is 1,2 then we turn them into list we get [1,2] and iloc takes 1st row 1st column, which is 5. 

In [7]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

orders.csv на месте: True


In [9]:
%pip install pandas numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 12.9 MB/s  0:00:01eta 0:00:01
Note: you may need to restart the kernel to use updated packages.


In [10]:
import pandas as pd
import numpy as np

df = pd.read_csv("orders.csv")
# Готовая строка: суммы вида "15 990" записаны текстом. Убираем пробел и переводим в числа (подробно на уроке 6)
df["amount"] = pd.to_numeric(df["amount"].str.replace(" ", ""))
print(df.shape)

(5463, 8)


In [11]:
# Даты записаны текстом "2026-09-01 08:00", поэтому строки можно сравнивать
sep = df["order_date"] < "2026-10"     # сентябрь
octo = ~sep                            # октябрь
paid = df["status"] == "оплачен"
not_test = df["customer_id"] != "TEST"
SH = ["Шымкент", "шымкент", "Shymkent"]
print("сентябрь:", sep.sum(), "| октябрь:", octo.sum())

сентябрь: 2782 | октябрь: 2681


## [База] 1. Паспорт таблицы
Сколько строк и столбцов? Какие типы у столбцов? Сколько пропусков в `delivery_days`? Сколько разных значений в `city`, `category`, `status`, `customer_id`?
Напишите менеджеру три проблемы данных, которые видно уже сейчас, и чем каждая опасна для отчёта о выручке.

In [12]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5463 entries, 0 to 5462
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   order_id       5463 non-null   int64  
 1   order_date     5463 non-null   str    
 2   customer_id    5463 non-null   str    
 3   city           5463 non-null   str    
 4   category       5463 non-null   str    
 5   amount         5463 non-null   float64
 6   delivery_days  5303 non-null   float64
 7   status         5463 non-null   str    
dtypes: float64(2), int64(1), str(5)
memory usage: 341.6 KB


In [16]:
df.shape

(5463, 8)

In [23]:
total=len(df["order_id"])
delivery=df["delivery_days"].count()
missed=total-delivery
missed

np.int64(160)

In [25]:
#another approach
missed = df["delivery_days"].isna().sum()   
print(missed)

160


In [30]:
print(df["city"].nunique())
print(df["category"].nunique())
print(df["status"].nunique())
print(df["customer_id"].nunique())

14
5
2
2003


3 problems: 1) city names are not consistent 2) amount was stored as string 3) 160 data is missing in delivery days 

## [База] 2. Типичный заказ
Для оплаченных заказов без TEST: среднее, медиана, 25-й и 75-й перцентили суммы.
Какой чек «типичный»? Что честнее показать в отчёте, среднее или медиану, и почему?

In [32]:
df.head()

,order_id,order_date,customer_id,city,category,amount,delivery_days,status
0,101674,2026-09-01 08:00,C00681,Алматы,Одежда,8070.0,1.0,оплачен
1,101214,2026-09-01 08:00,C01210,Астана,Красота,3630.0,2.0,оплачен
2,102424,2026-09-01 08:00,C01408,Караганда,Дом,9120.0,3.0,оплачен
3,102333,2026-09-01 08:00,C01529,Алматы,Одежда,9350.0,2.0,оплачен
4,102564,2026-09-01 09:00,C00657,Астана,Дом,11010.0,1.0,оплачен


In [39]:
test=df[df["customer_id"] == "TEST"]


In [44]:
without_t=df[(df["customer_id"]!="TEST") & (df["status"] =="оплачен")]


In [46]:
mean=without_t.amount.mean()
print(mean)
q25 = without_t["amount"].quantile(0.25)
q75 = without_t["amount"].quantile(0.75)
print(q25, q75)


14672.391348704346
5880.0 18690.0


## [База] 3. Отмены по городам
Доля отменённых заказов: по всем заказам, в Шымкенте, в Алматы (все написания каждого города через `isin`).
Какой город выделяется и во сколько раз?

In [52]:
print((df["status"] == "отменён").mean())

0.1023247300018305


In [53]:
print(df["city"].unique())

<StringArray>
[   'Алматы',    'Астана', 'Караганда',   'Шымкент',    'алматы',    'Атырау',
   'шымкент',  'Shymkent',    'астана',    'Актобе',   'Astana ',    'АЛМАТЫ',
   ' Алматы',    'Almaty']
Length: 14, dtype: str


In [54]:
shym = df[df["city"].isin(["Шымкент", "шымкент", "Shymkent"])]  

shym_share = (shym["status"] == "отменён").mean()
print(shym_share * 100)    

20.398009950248756


In [59]:
almaty = df[df["city"].isin(["Алматы", "алматы", "АЛМАТЫ", " Алматы", "Almaty"])]
almaty_share=(almaty["status"] == "отменён").mean()
print(round(almaty_share*100,3))

7.861


## [Стандарт] 4. Октябрь против сентября по двум городам
Выручка оплаченных заказов без TEST за сентябрь и октябрь в Шымкенте и в Алматы, изменение в %.
Дубли пока не убираем. Где падение сильнее и во сколько раз? Что сказать менеджеру?

In [63]:
alm_names = ["Алматы", "алматы", "АЛМАТЫ", " Алматы", "Almaty"]
shym_names = ["Шымкент", "шымкент", "Shymkent"] 

sep  = df[(df["order_date"] < "2026-10") & (df["customer_id"] != "TEST") & (df["status"] == "оплачен")]
octo = df[(df["order_date"] >= "2026-10") & (df["customer_id"] != "TEST") & (df["status"] == "оплачен")]

# revenue = sum of amount, per city, per month
alm_sep = sep[sep["city"].isin(alm_names)]["amount"].sum()
alm_oct = octo[octo["city"].isin(alm_names)]["amount"].sum()
shym_sep = sep[sep["city"].isin(shym_names)]["amount"].sum()
shym_oct = octo[octo["city"].isin(shym_names)]["amount"].sum()

print("Алматы   сентябрь:", alm_sep, "| октябрь:", alm_oct)
print("Шымкент  сентябрь:", shym_sep, "| октябрь:", shym_oct)

# change in %
alm_change = (alm_oct - alm_sep) / alm_sep * 100
shym_change = (shym_oct - shym_sep) / shym_sep * 100

print(f"Алматы: {alm_change:.1f}%")
print(f"Шымкент: {shym_change:.1f}%")
print(f"Во сколько раз: {shym_change / alm_change:.2f}")

Алматы   сентябрь: 15523940.0 | октябрь: 14643020.0
Шымкент  сентябрь: 4949800.0 | октябрь: 4044010.0
Алматы: -5.7%
Шымкент: -18.3%
Во сколько раз: 3.22


## [Стандарт] 5. Исправьте код от AI
AI написал код «отменённые заказы Шымкента». В нём **две** ошибки: одна роняет код, вторая тихо даёт неверное число. Найдите обе и исправьте.
```python
cancel_sh = df[df["city"] == "Шымкент" & df["status"] == "отменён"]
print(len(cancel_sh))
```

In [ ]:
# 5

## [Стандарт] 6. Самые частые клиенты
Кто сделал больше всего заказов (`value_counts` по `customer_id`)? Покажите все заказы этого клиента (`loc` с маской, столбцы дата, город, сумма, статус).
Есть ли среди его заказов повторяющиеся `order_id`? Что это значит для выручки, которую мы считали весь урок?

In [ ]:
# 6

## [Челлендж] 7. Выбросы по правилу IQR
Для оплаченных заказов без TEST: `IQR = Q3 − Q1`, граница = `Q3 + 1.5 * IQR`. Сколько заказов выше границы, какая это доля и какая у них категория чаще всего?
Сравните с ДЗ 4: по z-оценке выбросов было 87. Почему правила дают разное число?

In [ ]:
# 7

## [Челлендж] 8. Доля отмен для каждого написания города одной строкой
Поделите две Series: число отменённых заказов по `city` и число всех заказов по `city` (обе через `value_counts`). Отсортируйте.
Pandas делит по ярлыкам: проверьте это. Доля отмен у «Шымкент», «шымкент» и «Shymkent» сильно разная. Сколько заказов у каждого написания?
Можно ли судить о городе по одному написанию? Что нужно сделать с городами до анализа (это тема урока 6)?

In [ ]:
# 8